# 02 - ICCU cleaning

Reproducible verification of the cleaning rules applied to ICCU data and of the normalization of library statuses.

### Reproducibility

This notebook documents and verifies the ICCU data cleaning phase.

The transformations are implemented in the scripts in the `scripts/` directory and use the RAW archives stored in `data/raw/`.

The notebook allows the main cleaning, normalization, and missing-value handling rules applied to the dataset to be checked.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

def project_root():
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd.parent]
    for c in candidates:
        if (c / "data" / "processed").exists() and (c / "metadata").exists():
            return c
    raise FileNotFoundError("Eseguire il notebook dalla root del repository o da notebooks/.")

ROOT = project_root()
ROOT

## ICCU cleaning
Checking identifiers, dates, cleaned coordinates, missing values, and status normalization.

In [2]:
lib = pd.read_csv(ROOT / "data/processed/library.csv", dtype=str, keep_default_na=False, low_memory=False)
status = pd.read_csv(ROOT / "data/processed/library_status.csv", dtype=str, keep_default_na=False)
mapping = pd.read_csv(ROOT / "metadata/status_mapping.csv", dtype=str, keep_default_na=False)
cleanlog = pd.read_csv(ROOT / "reports/cleaning_log.csv", dtype=str, keep_default_na=False)
print(f"Biblioteche: {len(lib):,}")
print(f"Righe cleaning log: {len(cleanlog):,}")

Biblioteche: 19,611
Righe cleaning log: 10


In [3]:
assert lib["isil"].str.match(r"^IT-[A-Z]{2}\d{4}$").all()
flags = lib["coordinate_quality_flag"].value_counts(dropna=False)
flags

coordinate_quality_flag
valid_world_range               19521
zero_pair_treated_as_missing       63
missing                            24
outside_italy_bbox_review           3
Name: count, dtype: int64

In [4]:
expected = {
    "NESSUNO_STATO_SPECIALE_REGISTRATO": 13200,
    "BIBLIOTECA_NON_PIU_ESISTENTE": 1827,
    "BIBLIOTECA_NON_CENSITA": 1723,
    "BIBLIOTECA_CONFLUITA": 1502,
    "ALTRO_ISTITUTO_COLLEGATO_ICCU": 655,
    "TEMPORANEAMENTE_CHIUSA": 619,
    "BIBLIOTECA_IN_VIA_DI_ALLESTIMENTO": 34,
    "DEPOSITO_SENZA_PUNTO_DI_SERVIZIO": 33,
    "SERVIZI_SOSPESI_CAUSA_SISMA": 12,
    "INAGIBILE": 4,
    "RIAPERTURA_AGIBILITA_PARZIALE": 2,
}
actual = status["normalized_status"].value_counts().to_dict()
assert actual == expected
assert (status["include_in_main_analysis"] == "True").sum() == 2497
pd.Series(actual, name="count").rename_axis("normalized_status").to_frame()

,count
normalized_status,
NESSUNO_STATO_SPECIALE_REGISTRATO,13200
BIBLIOTECA_NON_PIU_ESISTENTE,1827
BIBLIOTECA_NON_CENSITA,1723
BIBLIOTECA_CONFLUITA,1502
ALTRO_ISTITUTO_COLLEGATO_ICCU,655
TEMPORANEAMENTE_CHIUSA,619
BIBLIOTECA_IN_VIA_DI_ALLESTIMENTO,34
DEPOSITO_SENZA_PUNTO_DI_SERVIZIO,33
SERVIZI_SOSPESI_CAUSA_SISMA,12


**Methodological note:** `NESSUNO_STATO_SPECIALE_REGISTRATO` does not mean that the library is certainly open; it preserves the absence of a special ICCU status.